# Retrieval over a graph

**Optional depth track · module 6 of 6**

**Goal:** Build a graph out of a syllabus, walk it, and decide **per question** whether the graph or plain keyword search should answer. Then say, with numbers from your own run, where the graph earned its keep and where it did not.

**Why it matters:** "Add a graph" gets sold as an upgrade to retrieval. It is not an upgrade, it is a different question. A graph answers what connects to what and what is reachable from where. Keyword search answers what a document says. Send a question to the wrong side and you get an answer that is confidently the wrong shape — and you pay for a schema and an extraction step whether or not it earned anything.

Nothing here is graded and nothing in the fifteen sessions depends on it. Work through it when you want the layer underneath.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

In [ ]:
# depth_checks registers this track's checkers. check() and review() are the
# same ones the course uses.
from bootcamp_agent.checks import check, review
import bootcamp_agent.depth_checks  # noqa: F401

## The fixture: one syllabus, two shapes

`fixtures/syllabus.json` is a made-up curriculum: fourteen concepts, eight sessions, and a
`depends_on` list per concept. It describes no real course and no real person.

The same file loads twice, into two shapes:

- **a graph** — `dict[str, list[str]]`, a concept id mapped to the ids it depends on. That is the
  whole data structure. There is no library here on purpose: an adjacency dict is what a property
  graph is underneath, and writing one is part of knowing what it costs;
- **documents** — one note per concept, handed to `bootcamp_agent.retrieval.retrieve`, the same
  lexical search the course uses. That is the baseline the graph has to beat.

In [ ]:
import json
from pathlib import Path

from bootcamp_agent.documents import Document
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.retrieval import retrieve

FIXTURES = next(p for p in (Path("fixtures"), Path("../fixtures")) if p.is_dir())
syllabus = json.loads((FIXTURES / "syllabus.json").read_text(encoding="utf-8"))
sessions = {s["id"]: s for s in syllabus["sessions"]}

# Shape 1: the graph. A concept id maps to the ids it depends on.
graph: dict[str, list[str]] = {c["id"]: list(c["depends_on"]) for c in syllabus["concepts"]}

# Shape 2: the same syllabus as text, for lexical search.
documents = [
    Document(
        doc_id=c["id"],
        title=c["name"],
        text=f"{c['name']}. Taught in session {int(c['taught_in'].split('-')[1])}.\n\n{c['note']}",
        source=f"syllabus.json#{c['id']}",
        tags=(c["taught_in"],),
    )
    for c in syllabus["concepts"]
]

edges = sum(len(parents) for parents in graph.values())
roots = sorted(concept for concept, parents in graph.items() if not parents)
print(f"{len(graph)} concepts, {edges} edges, {len(documents)} notes")
print("nothing comes before:", ", ".join(roots))

## 1. A walk that does not overstate what it knows

**Context.** "What must I understand before session 8" is not a search. It is every concept
reachable by following `depends_on` until the edges run out. Three things make that a real
exercise and not a loop:

- it is **multi-hop**. One hop answers a different question — the next thing, not everything;
- a prerequisite graph **can hold a cycle**, because somebody wrote an edge backwards. A walk with
  no memory of where it has been does not come back;
- a concept that is **not in the graph** is not a concept with nothing before it. Return `[]` for
  both and "I have never heard of this" comes out looking like "you are ready to start".

**Instructions.**

1. `prerequisites_of(graph, concept)` returns every concept reachable through `depends_on`,
   **sorted**, and **without** `concept` itself.
2. Return `None` when `concept` is not a key in `graph`. Return `[]` when it is a key with nothing
   before it. Those are different answers and the check tests both.
3. Keep a `seen` set. The check runs your function on a graph that has a cycle in it.

In [ ]:
def prerequisites_of(graph: dict[str, list[str]], concept: str) -> list[str] | None:
    """Every concept reachable through depends_on, sorted, excluding `concept` itself."""
    if concept not in graph:
        return None
    # TODO(you): this looks one hop and has no memory. Walk until the edges run out,
    # skip anything already seen, and keep `concept` out of its own answer.
    return sorted(graph[concept])


for concept in ("tokens", "caching", "fine-tuning"):
    print(f"{concept:14} {prerequisites_of(graph, concept)}")

**Expected output**

```
tokens         []
caching        ['chunking', 'cost-tracking', 'embeddings', 'lexical-search', 'retrieval', 'tokens']
fine-tuning    None
✅ d6-e1 passed
```

`[]` and `None` are the same length on screen and they are not the same claim.

In [ ]:
check("d6-e1", prerequisites_of)

## 2. The router: which side gets the question

**Context.** There are now two ways to answer, good at different things. The graph holds ids and
edges and not one sentence of prose. The notes hold prose and no edges at all. The router is one
small agent that reads a question and picks, and it has to say why, because a router you cannot
argue with is a coin flip you cannot debug.

Two cues do most of the work:

- **link words** — depend, before, prerequisite, affected, path, downstream. These ask what
  connects to what;
- **wording words** — say, note, explain, describe, quote. These ask what a document contains.

And one precedence rule, which is the actual decision: **wording beats links.** "What does the
grounding note say about what it depends on" names a relationship and still wants a sentence, and
the graph has no sentences to give it.

**Instructions.**

1. `route(question)` returns `{"tool": "graph" | "lexical", "why": "..."}`.
2. The check runs its own nine questions. Several look like relationship questions and are not:
   they carry a concept name, a session number, or a link word, and still ask what a note says.
   A router keyed on one word gets those wrong, and a router that always answers "graph" fails
   outright.
3. `why` names what about **this** question picked the side. The check refuses the same reason
   on both routes.

In [ ]:
LINK_WORDS = ("depend", "before", "prerequisite", "affect", "path", "downstream")
WORDING_WORDS = ("say", "note", "explain", "describe", "quote")


def route(question: str) -> dict:
    """Pick the side that can answer, and say what in the question picked it."""
    lowered = question.lower()
    # TODO(you): find the cues in `lowered`, apply the precedence rule, and build the
    # reason out of the cue you actually found rather than out of a constant.
    return {"tool": "graph", "why": ""}


for question in (
    "what must I understand before session 8",
    "if chunking changes, what else is affected",
    "which note explains a fixed vocabulary",
    "what does the grounding note say about what it depends on",
):
    verdict = route(question)
    print(f"{verdict['tool']:8} {question}")
    print(f"{'':8} -> {verdict['why'] or '(no reason given)'}")

**Expected output**

```
graph    what must I understand before session 8
         -> 'before' names an edge, and following edges is what search cannot do
graph    if chunking changes, what else is affected
         -> 'affect' names an edge, and following edges is what search cannot do
lexical  which note explains a fixed vocabulary
         -> 'note' asks what a note says, and the graph stores ids, not prose
lexical  what does the grounding note say about what it depends on
         -> 'say' asks what a note says, and the graph stores ids, not prose
✅ d6-e2 passed
```

The last one is the precedence rule doing its job.

In [ ]:
check("d6-e2", route)

## Two runs, and the numbers exercise 3 asks for

Everything in the next two cells is written for you. Read it, run it, keep the numbers.

### The extraction agent, and the edge it invented

Fourteen concepts and nineteen edges fit in a JSON file because somebody typed them. A real
syllabus arrives as prose, so the edges get pulled out by a small agent: a sentence goes in,
candidate `depends_on` edges come out. Here that agent is `FakeLLM`, so the run is offline and
identical every time — and it is scored against the edges the fixture already holds, which is the
only reason its mistakes are visible at all. An extraction step you cannot score is a step you are
trusting.

Watch what it misses, what it invents, and what the one invented edge does downstream.

> If exercise 1 is still unfinished, the two "prerequisites of chunking" numbers below come out
> small: a one-hop walk cannot see the damage.

In [ ]:
SENTENCES = [
    "Retrieval builds on lexical search and on embeddings.",
    "You cannot chunk anything until you understand tokens.",
    "Grounding needs retrieval, and it needs structured output to carry the citations.",
    "The agent loop assumes tool calling and grounding are already in place.",
    "Chunking decisions get revisited at deployment.",
    "Caching only makes sense once cost tracking exists.",
]

# The edges those six sentences actually state, read off the fixture by hand. This is
# the truth set, and without one the agent's output is just more text.
TRUE_EDGES_IN_SENTENCES = {
    ("retrieval", "lexical-search"),
    ("retrieval", "embeddings"),
    ("chunking", "tokens"),
    ("grounding", "retrieval"),
    ("grounding", "structured-output"),
    ("agent-loop", "tool-calling"),
    ("agent-loop", "grounding"),
    ("caching", "cost-tracking"),
}

extractor = FakeLLM(
    responses={
        "builds on lexical": '{"edges": [["retrieval", "lexical-search"], ["retrieval", "embeddings"]]}',
        "cannot chunk anything": '{"edges": [["chunking", "tokens"]]}',
        "carry the citations": '{"edges": [["grounding", "retrieval"]]}',
        "assumes tool calling": '{"edges": [["agent-loop", "tool-calling"], ["agent-loop", "grounding"]]}',
        "revisited at deployment": '{"edges": [["chunking", "deployment"]]}',
        "once cost tracking": '{"edges": [["caching", "cost-tracking"]]}',
    },
    default='{"edges": []}',
)

proposed = []
for sentence in SENTENCES:
    reply = extractor.complete("Extract prerequisite edges as JSON.", sentence)
    proposed.extend(tuple(edge) for edge in json.loads(reply)["edges"])

truth = {(node, parent) for node, parents in graph.items() for parent in parents}
correct = [edge for edge in proposed if edge in truth]
invented = [edge for edge in proposed if edge not in truth]
missed = sorted(TRUE_EDGES_IN_SENTENCES - set(proposed))

print(f"proposed {len(proposed)} edges from {len(SENTENCES)} sentences, {len(correct)} correct")
print(f"precision {len(correct) / len(proposed):.3f}    recall {len(correct) / len(TRUE_EDGES_IN_SENTENCES):.3f}")
print(f"invented: {invented}")
print(f"missed:   {missed}")


def find_cycle(graph: dict[str, list[str]]) -> list[str] | None:
    """Return one cycle as a path, or None. A schema you do not check is a schema you hope for."""
    state: dict[str, str] = {}
    path: list[str] = []

    def walk(node: str) -> list[str] | None:
        state[node] = "open"
        path.append(node)
        for parent in graph.get(node, []):
            if state.get(parent) == "open":
                return path[path.index(parent):] + [parent]
            if parent not in state:
                found = walk(parent)
                if found:
                    return found
        path.pop()
        state[node] = "done"
        return None

    for node in graph:
        if node not in state:
            found = walk(node)
            if found:
                return found
    return None


# Merge the agent's edges into the curated graph, the way anyone would.
merged = {node: list(parents) for node, parents in graph.items()}
for node, parent in proposed:
    if parent not in merged[node]:
        merged[node].append(parent)

before = prerequisites_of(graph, "chunking") or []
after = prerequisites_of(merged, "chunking") or []
print(f"\nprerequisites of 'chunking': {len(before)} curated -> {len(after)} after the merge")
print(f"cycle in the curated graph: {find_cycle(graph)}")
print(f"cycle in the merged graph:  {' -> '.join(find_cycle(merged) or ['none'])}")

### The comparison

Six probes: three about relationships, three about wording. **Both sides answer all six.** A probe
counts as answered only when everything it asked for is in the answer — every concept id for a
relationship probe, the phrase itself for a wording probe. One rule, applied to both sides, and
the required answers are written down here rather than read back out of whichever side produced
them.

Note what the graph needs and search does not: somebody has to say which traversal each question
wants. That step is exercise 2, and it is part of the bill.

In [ ]:
def dependents_of(graph: dict[str, list[str]], concept: str) -> list[str] | None:
    """Downstream is the same walk, on the reversed edges."""
    reverse: dict[str, list[str]] = {node: [] for node in graph}
    for node, parents in graph.items():
        for parent in parents:
            reverse.setdefault(parent, []).append(node)
    return prerequisites_of({node: sorted(kids) for node, kids in reverse.items()}, concept)


taught_in_8 = sessions["session-08"]["teaches"]
before_session_8 = sorted(
    {c for taught in taught_in_8 for c in (prerequisites_of(graph, taught) or [])} - set(taught_in_8)
)

PROBES = [
    {
        "question": "what must I understand before session 8",
        "graph_answer": before_session_8,
        "required": ["agent-loop", "chunking", "embeddings", "grounding", "lexical-search",
                     "prompting", "retrieval", "structured-output", "tokens", "tool-calling"],
    },
    {
        "question": "which concepts depend on retrieval",
        "graph_answer": dependents_of(graph, "retrieval") or [],
        "required": ["agent-loop", "caching", "deployment", "evaluation", "grounding"],
    },
    {
        "question": "if chunking changes, what else is affected",
        "graph_answer": dependents_of(graph, "chunking") or [],
        "required": ["agent-loop", "caching", "deployment", "embeddings", "evaluation",
                     "grounding", "lexical-search", "retrieval"],
    },
    {
        "question": "what does the chunking note say about paragraph boundaries",
        "graph_answer": prerequisites_of(graph, "chunking") or [],
        "required": ["paragraph boundaries"],
    },
    {
        "question": "which note explains a fixed vocabulary",
        "graph_answer": [],  # the question names no concept to walk from
        "required": ["fixed vocabulary"],
    },
    {
        "question": "what does the retrieval note say about ranking",
        "graph_answer": prerequisites_of(graph, "retrieval") or [],
        "required": ["ranking badly"],
    },
]


def answered(text: str, required: list[str]) -> bool:
    flat = text.replace("-", " ").lower()
    return all(item.replace("-", " ").lower() in flat for item in required)


graph_hits = lexical_hits = 0
for probe in PROBES:
    passages = retrieve(probe["question"], documents, top_k=3)
    by_graph = answered(" ".join(probe["graph_answer"]), probe["required"])
    by_lexical = answered(" ".join(hit.chunk.text for hit in passages), probe["required"])
    graph_hits += by_graph
    lexical_hits += by_lexical
    print(f"{probe['question'][:56]:58} graph {'hit ' if by_graph else 'miss'}   lexical {'hit ' if by_lexical else 'miss'}")

print(f"\ngraph answered {graph_hits}/{len(PROBES)}, lexical answered {lexical_hits}/{len(PROBES)}")

## 3. The comparison, with the losses left in

**Context.** You have both sides measured on the same six probes, and you know what the graph cost
to build and to keep. Write it down as a decision about **this** assistant, not as an opinion
about graphs.

**Instructions.**

1. `graph_won` and `graph_lost` each carry a number from the run above. The check refuses either
   one without a number, and refuses a `graph_lost` that opens by saying the graph never lost.
2. `maintenance_cost` is what keeping it correct costs: the edges somebody maintains by hand, what
   the extraction agent got wrong, and what that one wrong edge did to an answer.
3. `worth_keeping` is `True` or `False`. Commit to one.

In [ ]:
report = {
    "graph_won": "",          # TODO(you): where it beat search, with a number
    "graph_lost": "",         # TODO(you): where search beat it, with a number
    "maintenance_cost": "",   # TODO(you): edges to maintain, extraction errors, the damage
    "worth_keeping": None,    # TODO(you): True or False, for this assistant
}
for field, value in report.items():
    print(f"{field:18} {value if value not in ('', None) else '(empty)'}")

**Expected output**

```
graph_won          3 of 3 relationship probes, against 0 of 3 for lexical. ...
graph_lost         0 of 3 wording probes, against 3 of 3 for lexical. ...
maintenance_cost   19 edges somebody keeps correct. The extraction agent proposed 8 ...
worth_keeping      True
✅ d6-e3 passed
```

`worth_keeping: False` passes too. The check judges whether you measured, not which way you went.

In [ ]:
check("d6-e3", report)

## Review

The scorecard for this module. Every ❌ names the exercise and the hint.

In [ ]:
review("d6")